## 2. Training Data Prep

This is done in two phases, Lower Funnel (Last-Click ND) and Upper Funnel (MEI ND).

The results are grouped br Weekly date granularity, with Sundays starting the media week.

Our main predictor variable is Spend, and the response variable is Net Demand (configurable in functions/config.py).

Dependencies are fiscal_calendar and promo_dummies files.

### Lower Funnel

- Source: Tableau/Redshift BR Daily Actuals
- Metrics: Spend, LCND
- Channels:
    - All Affiliates (Content, Coupon, Loyalty ..)
    - All Search (Brand, NonBrand, PLA, PMAX, RSC ..)
    - Social CVR

### Upper Funnel 

- Source: MMM Weekly Actuals (Seawalls)
- Metrics: Spend, Online MEI ND
- Channels:
    - Social AWE, CON
    - Video AWE, CON

In [1]:
import pandas as pd
import numpy as np

In [2]:
# notebook config
from pathlib import Path
import sys
sys.path.append(str(Path.cwd().parent))

# packages
import pandas as pd

# functions
import functions.lr_models as lr
import functions.transform as tf

In [3]:
# config
config = {
    'run': '202609_3P', # unique name of the run, e.g. '2025-01'
}

# 202603_1P
# 202603_2P
# 202603_3P
# 202603_reflows
# 202604_dev
# 202604_mmm
# 202604_1P
# 202604_2P
# 202604_3P
# 202605_1P
# 202605_2P

## Lower Funnel
----

In [4]:
# process lower-funnel

# load data exported from Tableau as a CSV format, make sure this is the name of the file: BR Pacing Actuals.csv

# handling for tab-separated values

data_lf = tf.clean_df(
    pd.read_csv(
        '../data/raw/BR Pacing Actuals_20260822.csv',
        sep='\t',
        encoding='utf-16'))

# handling for comma-separated values

# data_lf = tf.clean_df(
#     pd.read_csv(
#         '../data/raw/BR Pacing Actuals_20260815.csv',
#         encoding='utf-8'))

print(data_lf.shape)

(36094, 17)


In [5]:
## columns
data_lf = data_lf[['date', 'brand', 'channel', 'funnel', 'tactic', 'spend', 'lcnd', 'visit']]
cols = ['date', 'brand', 'channel', 'funnel', 'tactic', 'spend', 'nd', 'visits']
data_lf.columns = cols

In [6]:
## dates 
data_lf['date'] = pd.to_datetime(data_lf['date'])

# calculate weekstart and fiscal month
data_lf['weekstart'] = data_lf['date'] - pd.to_timedelta((data_lf['date'].dt.dayofweek + 1) % 7, unit='D')

In [7]:
## brand
data_lf['brand'] = data_lf['brand'].apply(lambda x: x.lower())

# qa
print(data_lf['brand'].unique())

<ArrowStringArray>
['brand us', 'brand ca', 'brand outlet']
Length: 3, dtype: str


In [8]:
## funnel
# qa
print(data_lf['funnel'].unique())

<ArrowStringArray>
['Conversion', 'Awareness', 'Consideration']
Length: 3, dtype: str


In [9]:
## tactic
# data_lf['tactic'] = data_lf['tactic'].str.replace(r'\b(Test|Credits|Credit)$', '', regex=True) # aggregating 'Test' and 'Credit' versions
# data_lf['tactic'] = data_lf['tactic'].str.strip()


data_lf['tactic'] = data_lf['tactic'].str.replace(r'\b(Test|Credits|Credit)$', '', regex=True) # aggregating 'Test' and 'Credit' versions
data_lf['tactic'] = data_lf['tactic'].str.strip()

for phrase in [
    'Acquire (BAU)',
    'Acquire & Reclaim (BAU)',
    'Retention',
    'Stores',
    'Smart Plus',
    'Reels Trending Ads',
    ' CUS',
    ' DYN',
    ' MAX',
    'Search',
    'View Content',
]:
    data_lf['tactic'] = data_lf['tactic'].str.replace(phrase, '', regex=False)

# normalize spacing and remove any remaining stray separators
# e.g. 'Meta AWE Acquire & Reclaim' -> 'Meta AWE & Reclaim'
data_lf['tactic'] = data_lf['tactic'].str.replace(r'\s*&\s*', ' & ', regex=True)
data_lf['tactic'] = data_lf['tactic'].str.replace(r'\s{2,}', ' ', regex=True).str.strip()


# qa
df_schema = data_lf[['brand', 'channel','funnel','tactic']].drop_duplicates().sort_values(by=['brand', 'channel','funnel','tactic'])

print(data_lf['tactic'].unique())

<ArrowStringArray>
[    'Not Applicable',              'Brand',            'CVR DAB',
            'CVR DPA',           'Meta AWE',          'Non Brand',
      'Pinterest AWE',                'PLA',         'TikTok AWE',
            'CVR ASC',                'CVR',               'PMAX',
                'RSC', 'YouTube Shorts AWE',           'Meta CON',
      'Pinterest CON',         'TikTok CON',     'Ad Marketplace',
        'Aff Content',         'Aff Coupon',        'Aff Loyalty',
     'CVR ASC Volume',       'CVR ASC Omni',      'CVR ASC Value',
         'TikTok CVR']
Length: 25, dtype: str


In [10]:
## channel

# qa
data_lf['channel'].unique()

<ArrowStringArray>
['Affiliates', 'Search', 'Social']
Length: 3, dtype: str

In [11]:
# filter rows to keep lower funnel only
data_lf = data_lf[
    (data_lf['channel'].isin(['Affiliates', 'Search'])) |
    ((data_lf['channel'] == 'Social') & (data_lf['funnel'] == 'Conversion'))]

# qa
df_schema = data_lf[['brand', 'channel','funnel','tactic']].drop_duplicates().sort_values(by=['brand', 'channel','funnel','tactic'])

In [12]:
# numeric values
# remove currency, handle parentheses for negative values, and convert to decimal
def to_numeric(s):
    return s.replace('[\$,]', '', regex=True).replace(r'\((.*)\)', r'-\1', regex=True).fillna(0).astype(float).round(4)

for col in ['spend', 'nd', 'visits']:
    data_lf[col] = to_numeric(data_lf[col])

In [13]:
## filter out custom ranges or conditions as needed

# remove BRSP Search CVR Non Brand, greater instability after this point.  2026-03-10
data_lf = data_lf[~(
    (
        (data_lf['brand'] == 'brand us') & 
        (data_lf['channel'] == 'Search') & 
        (data_lf['funnel'] == 'Conversion') &
        (data_lf['tactic'] == 'Non Brand') & 
        (data_lf['date'] < '2025-07-06')
    ) |
    (
        (data_lf['brand'] == 'brand us') & 
        (data_lf['channel'] == 'Social') & 
        (data_lf['funnel'] == 'Conversion') &
        (data_lf['tactic'] == 'CVR ASC Omni') & 
        (data_lf['date'] < '2025-08-10')
    ) |
    (data_lf['spend'] == 0)
)]

In [14]:
# model label
data_lf['model'] = data_lf[['channel', 'funnel', 'tactic']].agg(' | '.join, axis=1)

In [15]:
# date grouping
calendar = tf.clean_df(pd.read_csv(f"../data/meta/fiscal_calendar.csv"))
calendar['weekstart'] = pd.to_datetime(calendar['weekstart'], format='%m/%d/%y')

# merge in fyear and fmonth
data_lf = data_lf.merge(calendar[['weekstart','fyear', 'fmonth']], on='weekstart', how='left')

In [16]:
# group by week
data_lf_weekly = data_lf.groupby(['brand', 'model','weekstart'], as_index=False)[['spend', 'nd', 'visits']].sum()

# be sure to manually QA
data_lf_weekly.to_csv(f"../data/qa/data_lf_{config['run']}.csv")
data_lf_weekly.head(5)

,brand,model,weekstart,spend,nd,visits
0,brand us,Affiliates | Consideration | Aff Content,2025-05-04,33070.96,218413.57,88052.0
1,brand us,Affiliates | Consideration | Aff Content,2025-05-11,35298.67,258936.68,97515.0
2,brand us,Affiliates | Consideration | Aff Content,2025-05-18,38541.76,263891.69,103692.0
3,brand us,Affiliates | Consideration | Aff Content,2025-05-25,38554.49,265505.21,90449.0
4,brand us,Affiliates | Consideration | Aff Content,2025-06-01,37186.85,182437.52,69795.0


### Upper Funnel
----

In [17]:
# load data
data_uf_brsp = pd.read_excel('../data/raw/BR Media Weekly Spend & Incremental Net Demand & Traffic - FebFY26.xlsx', sheet_name='BRSP', skiprows=3).iloc[:, [1,2,3,4,7,12]]

data_uf_brfs = pd.read_excel('../data/raw/BR Media Weekly Spend & Incremental Net Demand & Traffic - FebFY26.xlsx', sheet_name='BRFS', skiprows=3).iloc[:, [1,2,3,4,7,12]]

data_uf_brca = pd.read_excel('../data/raw/BRCA Media Weekly Spend & Incremental Net Demand, Acquisition, Traffic - YTD25.xlsx', sheet_name='BRCA SP', skiprows=3).iloc[:, [1,2,3,4,7,12]]

# brand
data_uf_brsp.insert(0, 'brand', 'brand us')
data_uf_brfs.insert(0, 'brand', 'brand outlet')
data_uf_brca.insert(0, 'brand', 'brand ca')

# concat
data_uf = tf.clean_df(pd.concat([data_uf_brsp, data_uf_brfs, data_uf_brca], axis=0, ignore_index=True))

# columns
cols = ['brand', 'funnel', 'channel', 'date', 'spend', 'nd', 'visits']
data_uf.columns = cols

# qa
data_uf.groupby(['brand', 'funnel', 'channel']).size().reset_index(name='count')

,brand,funnel,channel,count
0,brand us,Awareness,AWE Audio,56
1,brand us,Awareness,AWE Digital Video Online: Disney,56
2,brand us,Awareness,AWE Digital Video Online: Non YT Shorts,56
3,brand us,Awareness,AWE Digital Video Online: YT Shorts,56
4,brand us,Awareness,AWE OOH,56
...,...,...,...,...
100,brand outlet,Conversion,CVR Social Meta DPA,56
101,brand outlet,Other,OTH Loyalty Social,56
102,brand outlet,Owned,OWN Direct Mail,56
103,brand outlet,Owned,OWN Email,56


In [18]:
# dates

# date in datetime format
data_uf['date'] = pd.to_datetime(data_uf['date'], format='%m/%d/%y')

# calculate weekstart -- Seawalls gives end-of-week (Sat) so get prior Sunday
data_uf['weekstart'] = data_uf['date'] - pd.to_timedelta((data_uf['date'].dt.dayofweek + 1) % 7, unit='D')

In [19]:
# funnel

# filter out Owned etc.
data_uf = data_uf[data_uf['funnel'].isin(['Awareness','Consideration','Conversion'])]

# qa
data_uf['funnel'].unique()

<ArrowStringArray>
['Awareness', 'Consideration', 'Conversion']
Length: 3, dtype: str

In [20]:
# channel

# filter ut AWE/CON channels either not modeled or that use LC
data_uf = data_uf[(~data_uf['channel'].str.contains('Audio|OOH|Print|Publisher|Affiliate|Search|PLA|PMAX|RSC|Brand|NB', case=False, na=False))]

# QA
data_uf[['funnel', 'channel']].drop_duplicates()

,funnel,channel
56,Awareness,AWE Digital Video Online: Disney
112,Awareness,AWE Digital Video Online: Non YT Shorts
168,Awareness,AWE Digital Video Online: YT Shorts
392,Awareness,AWE Social Meta
448,Awareness,AWE Social Meta Influencer
504,Awareness,AWE Social Meta RTN
560,Awareness,AWE Social Pinterest
616,Awareness,AWE Social TikTok
672,Awareness,AWE Social TikTok Influencer
784,Consideration,CON Digital Video Online: Non-YT Shorts


In [21]:
# channel and tactic

data_uf['tactic'] = data_uf['channel']

conditions = [
    data_uf['channel'].str.contains('Social', case=False, na=False),
    data_uf['channel'].str.contains('Video', case=False, na=False),
]

choices = ['Social', 'Video']

data_uf['channel'] = np.select(conditions, choices, default=data_uf['channel'])

# qa
data_uf[['brand', 'channel', 'funnel', 'tactic']].drop_duplicates()

,brand,channel,funnel,tactic
56,brand us,Video,Awareness,AWE Digital Video Online: Disney
112,brand us,Video,Awareness,AWE Digital Video Online: Non YT Shorts
168,brand us,Video,Awareness,AWE Digital Video Online: YT Shorts
392,brand us,Social,Awareness,AWE Social Meta
448,brand us,Social,Awareness,AWE Social Meta Influencer
504,brand us,Social,Awareness,AWE Social Meta RTN
560,brand us,Social,Awareness,AWE Social Pinterest
616,brand us,Social,Awareness,AWE Social TikTok
672,brand us,Social,Awareness,AWE Social TikTok Influencer
784,brand us,Video,Consideration,CON Digital Video Online: Non-YT Shorts


In [22]:
# clean and order

# data_uf['channel'] = data_uf['channel'] + ' ' + data_uf['funnel'] + ' ' + data_uf['platform'] + ' ' + data_uf['tactic']
# data_uf = data_uf[['date','weekstart','brand','channel','spend','nd']].sort_values(by=['brand','channel','date','weekstart'])

# qa
# data_uf.to_csv('../data/qa/data_uf.csv'))

In [23]:
# numeric values
# spend and nd; remove currency, handle parentheses for negative values, and convert to decimal
for col in ['spend', 'nd', 'visits']:
    data_uf[col] = to_numeric(data_uf[col])

# remove rows with no data
data_uf = data_uf[~((data_uf['spend'] == 0) & (data_uf['nd'] == 0))]

In [24]:
# filter out custom ranges or conditions as needed

# filter out due to Seawalls proxy changes 2026-02-01
data_uf = data_uf[~(
    (data_uf['brand'] == 'brand outlet') &
    (data_uf['channel'] == 'Social') &
    (data_uf['funnel'] == 'Awareness') &
    (data_uf['tactic'] == 'AWE Social Meta') & 
    (data_uf['weekstart'] < '2025-01-05')
)]

# filter out rows with Net Demand but no Spend (assuming this is decay effects, short-term solution for now) 2026-03-11
data_uf = data_uf[~((data_uf['nd'] > 0) & (data_uf['spend'] == 0))]


In [25]:
# date grouping
calendar = tf.clean_df(pd.read_csv(f"../data/meta/fiscal_calendar.csv"))
calendar['weekstart'] = pd.to_datetime(calendar['weekstart'], format='%m/%d/%y')

# merge in fyear and fmonth
data_uf = data_uf.merge(calendar[['weekstart','fyear', 'fmonth']], on='weekstart', how='left')

In [26]:
# model label
data_uf['model'] = data_uf[['channel', 'funnel', 'tactic']].agg(' | '.join, axis=1)

In [27]:
# model hotfixes

# Canada Meta AWE and CON rename "..Custom"
data_uf['model'] = data_uf['model'].replace('Social | Awareness | AWE Social META Custom', 'Social | Awareness | AWE Social Meta')
data_uf['model'] = data_uf['model'].replace('Social | Consideration | CON Social META Custom', 'Social | Consideration | CON Social Meta')


In [28]:
# group by week
data_uf_weekly = data_uf.groupby(['brand', 'model', 'weekstart'], as_index=False)[['spend', 'nd', 'visits']].sum()

# be sure to manually QA
data_uf_weekly.to_csv(f"../data/qa/data_uf_{config['run']}.csv")
data_uf_weekly.head(5)

,brand,model,weekstart,spend,nd,visits
0,brand us,Social | Awareness | AWE Social Meta,2025-02-02,12501.4601,118679.1845,13795.6329
1,brand us,Social | Awareness | AWE Social Meta,2025-02-09,13166.4100,125842.6389,14617.6805
2,brand us,Social | Awareness | AWE Social Meta,2025-02-16,17410.7301,173416.8917,20157.0183
3,brand us,Social | Awareness | AWE Social Meta,2025-02-23,14551.6800,139022.3327,16130.5997
4,brand us,Social | Awareness | AWE Social Meta,2025-03-02,22963.3000,161703.7629,19188.0692


### Final Output

- Contact the Weekly data (upper + lower) and merge in promo dummies

In [29]:
# weekly with promo dummies appended
df_weekly = pd.concat([data_uf_weekly, data_lf_weekly])

df_promos = pd.read_csv(f"../data/train/promo_dummies.csv")
df_promos['weekstart'] = pd.to_datetime(df_promos['weekstart'])
df_weekly = df_weekly.merge(df_promos, on=['brand', 'weekstart'], how='left')

df_weekly.to_csv(f"../data/train/train_{config['run']}.csv", index=False)

# QA
df_weekly.head(5)

,brand,model,weekstart,spend,nd,visits,promo_presidents_day,promo_friends_&_family,promo_easter_pre-peak,promo_easter_peak,...,promo_winter_sale_phase_3,promo_easter,promo_labor_day,promo_winter_sale_preview,discount_0_5,discount_0_4,discount_0_7,discount_0_6,discount_0_2,discount_0_3
0,brand us,Social | Awareness | AWE Social Meta,2025-02-02,12501.4601,118679.1845,13795.6329,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,brand us,Social | Awareness | AWE Social Meta,2025-02-09,13166.4100,125842.6389,14617.6805,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,brand us,Social | Awareness | AWE Social Meta,2025-02-16,17410.7301,173416.8917,20157.0183,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,brand us,Social | Awareness | AWE Social Meta,2025-02-23,14551.6800,139022.3327,16130.5997,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,brand us,Social | Awareness | AWE Social Meta,2025-03-02,22963.3000,161703.7629,19188.0692,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [30]:
# list of available models - update the forecast workbook lookups
# df_models = pd.DataFrame(df_weekly[['brand','model']].drop_duplicates

df_models = df_weekly.drop_duplicates(subset=['brand', 'model'])[['brand','model']].sort_values(['brand','model'])